In [1]:
# import bibliotek i wczytanie danych
import pandas as pd
import numpy as np
import re
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

from flair.models import TextClassifier
from flair.data import Sentence
flair_model = TextClassifier.load("sentiment")

from nltk.sentiment import SentimentIntensityAnalyzer
import nltk
nltk.download('vader_lexicon')

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np

import os
import sys

from transformers import pipeline

from sklearn.metrics import accuracy_score, f1_score

df = pd.read_csv('english_bg3_reviews_no_spam.csv', sep=';', encoding='utf-8')
df = df.dropna(subset=['clean_review']).reset_index(drop=True) # wyrzucamy nulle
df.head()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\huber\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


,review_id,review,voted_up,author_steamid,date_str,review_length,review_word_count,clean_review,is_spam_label,num_emojis,low_quality,is_spam
0,209490463,The greatest game of all time. It simply is. \r\n,True,76561198842852247,2025-11-17 20:17:24,43,9,the greatest game of all time it simply is,0,0,0,0
1,209488275,The dude trying to bite me at night was messed...,True,76561198335095296,2025-11-17 19:45:37,55,12,the dude trying to bite me at night was messed...,0,0,0,0
2,209488128,"This is probably a great solo game, but my hop...",False,76561198031957731,2025-11-17 19:43:28,131,29,this is probably a great solo game but my hope...,0,0,0,0
3,209487979,One of the best RPG's of the last decade,True,76561198140742800,2025-11-17 19:41:16,39,9,one of the best rpgs of the last decade,0,0,0,0
4,209482559,Definitely one of the best games I've played i...,True,76561198024036703,2025-11-17 18:19:29,61,12,definitely one of the best games ive played in...,0,0,0,0


In [2]:
# tokenizacja recenzji

def tokenize(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    tokens = [w for w in text.split() if len(w) > 2]
    return tokens

df["tokens"] = df["clean_review"].apply(tokenize)
df["tokens"].head()


0             [the, greatest, game, all, time, simply]
1    [the, dude, trying, bite, night, was, messed, ...
2    [this, probably, great, solo, game, but, hope,...
3            [one, the, best, rpgs, the, last, decade]
4    [definitely, one, the, best, games, ive, playe...
Name: tokens, dtype: object

In [3]:
# słowniki sentymentu 
keyword_categories = {
    'grafika': ['graf', 'visual', 'graph', 'art', 'design', 'textures', 'lighting', 'shaders', 'models', 'animation', 'animations', 'resolution', '4k', 'hd', 'detail', 'detailed', 'details', 'aesthetic', 'aesthetics', 'style', 'visuals', 'render', 'rendering'],
    'soundtrack': ['sound', 'muzyk', 'music', 'audio', 'sfx', 'soundtrack', 'voice acting', 'va', 'voices', 'vo', 'sound design', 'ost', 'official soundtrack', 'ambient', 'sound quality', 'volume', 'sound effects'],
    'fabuła': ['fab', 'story', 'plot', 'narrat', 'questline', 'quests', 'main story', 'side story', 'side quests', 'lore', 'dialogue choices', 'scenario', 'writing', 'plot twist', 'plot twists'],
    'mechanika': ['mechan', 'gameplay', 'multiplay', 'singleplay', 'coop', 'co-op', 'combat', 'controls', 'crafting', 'skills', 'builds', 'exploration', 'movement', 'jumping'],
    'postacie': ['posta', 'charact', 'bohater', 'hero', 'cosplay', 'npc', 'villains', 'protagonist', 'antagonist', 'character desing', 'personality'],
    'dialogi': ['dialog', 'rozmow', 'funny', 'humor', 'comedy', 'humour', 'banter', 'conversation', 'jokes', 'sarcasm'],
    'świat': ['świat', 'world', 'setting', 'immers', 'environ', 'open world', 'map', 'locations', 'areas', 'biomes', 'exploration', 'atmosphere'],
    'klimat': ['klimat', 'atmos', 'vibe', 'mood', 'creepy', 'cozy', 'intense', 'tense', 'feeling'],
    'bugi': ['bug', 'glitch', 'crash', 'error', 'problem', 'issue', 'lag', 'freeze', 'unstable', 'not working', 'corrupted', 'bugs', 'buggy'],
    'optymalizacja': ['optymal', 'perform', 'fps', 'performance', 'loading time', 'cpu', 'gpu', 'ram', 'optimization'],
    'sterowanie': ['sterow', 'control', 'keyb', 'pad', 'mouse', 'controls', 'keybinds', 'controller', 'keyboard', 'shortcut', 'shortcuts', 'sensitivity'],
    'balans': ['balan', 'diff', 'eas', 'hard', 'challeng', 'easy', 'difficult', 'scaling', 'overpowered', 'op', 'nerf', 'challenge'],
    'interfejs': ['interfejs', 'interface', 'ui', 'hud', 'menu', 'layout', 'navigation'],
    'cena': ['cen', 'price', 'koszt', 'cost', 'worth it', 'worth', 'expensive', 'cheap', 'overpriced', 'deal', 'discount', 'sale']
}

print(keyword_categories)


{'grafika': ['graf', 'visual', 'graph', 'art', 'design', 'textures', 'lighting', 'shaders', 'models', 'animation', 'animations', 'resolution', '4k', 'hd', 'detail', 'detailed', 'details', 'aesthetic', 'aesthetics', 'style', 'visuals', 'render', 'rendering'], 'soundtrack': ['sound', 'muzyk', 'music', 'audio', 'sfx', 'soundtrack', 'voice acting', 'va', 'voices', 'vo', 'sound design', 'ost', 'official soundtrack', 'ambient', 'sound quality', 'volume', 'sound effects'], 'fabuła': ['fab', 'story', 'plot', 'narrat', 'questline', 'quests', 'main story', 'side story', 'side quests', 'lore', 'dialogue choices', 'scenario', 'writing', 'plot twist', 'plot twists'], 'mechanika': ['mechan', 'gameplay', 'multiplay', 'singleplay', 'coop', 'co-op', 'combat', 'controls', 'crafting', 'skills', 'builds', 'exploration', 'movement', 'jumping'], 'postacie': ['posta', 'charact', 'bohater', 'hero', 'cosplay', 'npc', 'villains', 'protagonist', 'antagonist', 'character desing', 'personality'], 'dialogi': ['dial

In [4]:
# poszerzanie ręcznie stworzonego słownika o często występujące w recenzjach słowa
all_words = Counter()
for tokens in df["tokens"]:
    all_words.update(tokens)

most_common_words = all_words.most_common(400)

def auto_assign_category(word, keyword_categories):
    word_lower = word.lower()
    for category, fragments in keyword_categories.items():
        for frag in fragments:
            frag_lower = frag.lower()
            if len(frag_lower) < 4:
                continue

            if frag_lower in word_lower:
                return category
    return None

new_keywords = {}
for word, count in most_common_words:
    cat = auto_assign_category(word, keyword_categories)
    if cat:
        if word not in keyword_categories[cat]:
            new_keywords.setdefault(cat, []).append(word)

for cat, words in new_keywords.items():
    keyword_categories[cat].extend(words)

print("Nowe słowa dodane do słownika kategorii:")
print(new_keywords)

Nowe słowa dodane do słownika kategorii:
{'postacie': ['characters', 'character'], 'balans': ['different'], 'mechanika': ['mechanics', 'multiplayer'], 'grafika': ['graphics'], 'dialogi': ['dialogue'], 'świat': ['immersive'], 'fabuła': ['storytelling', 'explore', 'narrative'], 'bugi': ['issues']}


In [5]:
# sentyment 1 - vader
sia = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    score = sia.polarity_scores(str(text))["compound"]
    if score >= 0.05: return 1
    elif score <= -0.05: return 0
    else: return 2  # neutral

df["sent_vader"] = df["clean_review"].apply(vader_sentiment)
df["sent_vader"].head()

0    1
1    0
2    1
3    1
4    1
Name: sent_vader, dtype: int64

In [6]:
# optymalizacja vader
# wcześniejsza komórka to podstawowy pakiet VADER i jego wyniki dla naszych recenzji
# teraz zrobimy kilka modyfikacji, aby poprawić wyniki dla naszego konkretnego zbioru danych
# tak aby wyciągnąć wszystko co najlepsze z tego pakietu
# porównanie będzie można zobaczyć w zbiorze danych, w kolumnach sent_vader i sent_vader_opt

from nltk.sentiment import SentimentIntensityAnalyzer

sia_opt = SentimentIntensityAnalyzer()

# dodajemy słowa żeby vader lepiej rozumiał ironie i inne ludzkie emocje, której nie wykrywa w podstawowej wersji
custom_words = {
    "masterpiece": 3.0,
    "breathtaking": 2.0,
    "peak": 1.5,
    "insane": 1.5,
    "garbage": -2.5,
    "trash": -2.5,
    "mid": -1.0,
}

for w, val in custom_words.items():
    sia_opt.lexicon[w] = val

# klasyfikacja będzie bardziej rygorystyczna, żeby uniknąć fałszywych pozytywów i negatywów

def vader_sentiment_optimized(text):
    score = sia_opt.polarity_scores(str(text))["compound"]

    if score >= 0.15:
        return 1   # pozytywna
    elif score <= -0.15:
        return 0   # negatywna
    else:
        return 2   # neutralna

df["sent_vader_opt"] = df["review"].apply(vader_sentiment_optimized)
df[["sent_vader", "sent_vader_opt"]].head(10)


,sent_vader,sent_vader_opt
0,1,1
1,0,0
2,1,1
3,1,1
4,1,1
5,0,0
6,0,0
7,2,2
8,1,1
9,1,1


In [7]:
# sentyment 2 - bert

# model i tokeny 
tokenizer = AutoTokenizer.from_pretrained("nlptown/bert-base-multilingual-uncased-sentiment")
model = AutoModelForSequenceClassification.from_pretrained("nlptown/bert-base-multilingual-uncased-sentiment")

# analiza sentymentu - pozytywne negatywne i neutralne (1-2; 3; 4-5)
def bert_sentiment(text):
    try:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
        with torch.no_grad():
            outputs = model(**inputs)

        probs = torch.softmax(outputs.logits, dim=1).numpy()[0]
        rating = np.argmax(probs) + 1  
        if rating <= 2:
            return 0
        elif rating == 3:
            return 2
        else:
            return 1

    except Exception as e:
        return None

df["bert_sentiment"] = df["clean_review"].apply(bert_sentiment)

df["bert_sentiment"].value_counts()


KeyboardInterrupt: 

In [7]:
# model działa, ale ze względu na ograniczenia sprzętowe jest bardzo wolny - ponad 360 minut pracy i brak wyniku!
# z tego względu model trzeba zoptymalizować - np. dzięki batchowaniu
# mamy około 80 tysięcy rekordów do przerobienia, więc trzeba wybrać dobry batch_size, tak żeby przyspieszyć pracę i jednocześnie
# go nie przeciążyć
# żeby przyspieszyć działanie modelu potrzebujemy ok. 2500 batchy po 32 recenzje (2500*32=80000 recenzji - zgadza się)
# to wstępna estymacja, która jeśli się nie sprawdzi to zostanie zmieniona 

tokenizer = AutoTokenizer.from_pretrained("nlptown/bert-base-multilingual-uncased-sentiment")
model = AutoModelForSequenceClassification.from_pretrained("nlptown/bert-base-multilingual-uncased-sentiment")
model.eval()

def bert_sentiment_batch(texts, batch_size=32):
    results = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        inputs = tokenizer(batch, return_tensors="pt", truncation=True, padding=True)
        with torch.no_grad():
            outputs = model(**inputs)

        probs = torch.softmax(outputs.logits, dim=1).numpy()
        ratings = np.argmax(probs, axis=1) + 1

        # sprowadzamy oceny gwiazdkowe berta 1-5 do 3 klas sentymentu tak żeby móc porównać z innymi modelami
        # tzn. 2 neutralne, 1 pozytywne i 0 negatywne
        mapped = [
            0 if r <= 2 else (2 if r == 3 else 1)
            for r in ratings
        ]
        results.extend(mapped)

    return results

df["bert_sentiment"] = bert_sentiment_batch(df["clean_review"].tolist(), batch_size=32)

In [8]:
df["bert_sentiment"].head()

0    1
1    0
2    0
3    1
4    1
Name: bert_sentiment, dtype: int64

In [9]:
# sentyment 3 - regresja logistyczna i wektory tfidf
tfidf = TfidfVectorizer(min_df=5, max_df=0.95, ngram_range=(1,2), max_features=60000)
X = tfidf.fit_transform(df["clean_review"])
y = df["voted_up"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

logreg = LogisticRegression(max_iter=5000, class_weight='balanced', n_jobs=-1)
logreg.fit(X_train, y_train)
y_pred_logreg = logreg.predict(X_test)

print("\n=== Logistic Regression REPORT ===")
print(classification_report(y_test, y_pred_logreg))


=== Logistic Regression REPORT ===
              precision    recall  f1-score   support

           0       0.50      0.84      0.63       541
           1       0.99      0.96      0.97     10069

    accuracy                           0.95     10610
   macro avg       0.75      0.90      0.80     10610
weighted avg       0.97      0.95      0.96     10610



In [10]:
# model radzi sobie z wartościami 1, ale kompletnie nie daje rady dla 0, dlatego wymaga optymalizacji żeby móc go porównać z innymi
# do optymalizacji wykorzystamy balansowanie klas - SMOTE

from imblearn.over_sampling import SMOTE

# poprawiamy wektory tfid, żeby zwiększyć accuracy modelu
tfidf_opt = TfidfVectorizer(
    min_df=3,
    max_df=0.97,
    ngram_range=(1,3),
    max_features=120000,
    sublinear_tf=True
)
X_opt = tfidf_opt.fit_transform(df["clean_review"])
y = df["voted_up"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X_opt, y, test_size=0.2, random_state=42, stratify=y
)

# SMOTE dla zbalansowania klas
sm = SMOTE(random_state=42)
X_train_bal, y_train_bal = sm.fit_resample(X_train, y_train)

# poprawa parametrów modelu regresji logistycznej
logreg_opt = LogisticRegression(
    solver="saga",
    penalty="l2",
    class_weight="balanced",
    C=1.5,
    max_iter=8000,
    n_jobs=-1,
)

logreg_opt.fit(X_train_bal, y_train_bal)

y_pred_opt = logreg_opt.predict(X_test)
print("\n=== Logistic Regression REPORT (OPTIMIZED) ===")
print(classification_report(y_test, y_pred_opt))


=== Logistic Regression REPORT (OPTIMIZED) ===
              precision    recall  f1-score   support

           0       0.61      0.74      0.67       548
           1       0.99      0.97      0.98     10062

    accuracy                           0.96     10610
   macro avg       0.80      0.86      0.82     10610
weighted avg       0.97      0.96      0.96     10610



In [11]:
# optymalizacja regresji logistycznej poprawiła wyniki dla obu klas
# ale to nadal za mało - model kompletnie nie radzi sobie z precyzją dla klasy 0 - daje zbyt dużo fałszywych pozytywów
# w związku z tym należy zmienić model regresji logistycznej na inny algorytm uczenia maszynowego - SVC
#SVC radzi sobie o wiele lepiej z wektorami tfidf i prawdopodobnie poprawi klasyfikację dla obu klas


# sentyment 3 - SVC z wektorami tfidf
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# SMOTE zeby zbalansowac klasy
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

param_grid = {
    "C": [0.1, 0.3, 0.5, 1, 2],
    "loss": ["hinge", "squared_hinge"],
    "class_weight": ["balanced", None],
    "max_iter": [50000]
}

svc_grid = GridSearchCV(
    LinearSVC(),
    param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=1
)

svc_grid.fit(X_train_bal, y_train_bal)

print("=== Best params ===")
print(svc_grid.best_params_)


y_pred_svm_opt = svc_grid.best_estimator_.predict(X_test)
print("\n=== SVC Optimized ===")
print(classification_report(y_test, y_pred_svm_opt))

Fitting 3 folds for each of 20 candidates, totalling 60 fits
=== Best params ===
{'C': 2, 'class_weight': 'balanced', 'loss': 'hinge', 'max_iter': 50000}

=== SVC Optimized ===
              precision    recall  f1-score   support

           0       0.73      0.57      0.64       548
           1       0.98      0.99      0.98     10062

    accuracy                           0.97     10610
   macro avg       0.85      0.78      0.81     10610
weighted avg       0.96      0.97      0.96     10610



c:\Users\huber\Desktop\inyznierka\python\venv\Lib\site-packages\sklearn\svm\_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [12]:
# SVC nie daje sobie rady z klasą 0 mimo optymalizacji, więc model zostaje odrzucony

In [12]:
# sentyment 4 - flair
from flair.models import TextClassifier
from flair.data import Sentence

# model dla sentymentu 3 klasowego - pozytywne (1), negatywne (0) i neutralne(2)
flair_model = TextClassifier.load("sentiment")

def flair_sentiment(text):
    text = str(text)

    if not text.strip():
        return 2  # funkcja bedzie przypisywac neutralny sentyment dla pustych opinii (chociaz nie powinno ich byc)

    sentence = Sentence(text)
    flair_model.predict(sentence)

    label = sentence.labels[0].value.upper()

    if label == "POSITIVE":
        return 1
    elif label == "NEGATIVE":
        return 0
    else:
        return 2   # opinia neutralna

df["sent_flair"] = df["clean_review"].apply(flair_sentiment)


'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 98ba51f7-d9d8-45fe-9bbf-893803705b5e)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


In [ ]:
# flair to model end to end, wiec nie trzeba go optymalizowac, tak jak vadera
# wyniki sentymentu flair z założenia nie wymagają optymalizacji i są "dobre"

In [5]:
# skategoryzowanie sentymentu na podstawie słownika

def categorize_review(text):
    found = set()
    text_lower = str(text).lower()
    for category, fragments in keyword_categories.items():
        for frag in fragments:
            if frag in text_lower:
                found.add(category)
    return ", ".join(sorted(found)) if found else "Unspecified"

df["review_category"] = df["clean_review"].apply(categorize_review)

print(df["review_category"].value_counts().head(10))



review_category
Unspecified           18235
balans                 3829
soundtrack             2120
grafika                1472
fabuła                 1253
cena                   1162
interfejs               685
balans, soundtrack      667
postacie                621
mechanika               562
Name: count, dtype: int64


In [6]:
# słowniki sentymentu 
expectation_categories = {
    "DLC": [
        "dlc", "expansion", "expansion pack", "more content", "new content", 
        "addon", "add-on", "bonus content", "season pass"],
    "Sequel": [
        "sequel", "next game", "next one", "bg4", "baldur's gate 4", 
        "follow-up", "continuation", "trilogy", "future game"],
    "Patchowanie": [
        "fix", "patch", "update", "needs fixing", "needs update", "glitch", 
        "bug fix", "crash fix", "optimization", "optimize", "repair", "broken"],
    "Co-op": [
        "multiplayer", "coop", "co-op", "online", "pvp", "pve", 
        "crossplay", "play with friends"],
    "Mechanika": [
        "new game plus", "ng+", "transmog", "customization", "photo mode", 
        "difficulty", "hard mode", "easy mode", "accessibility", "feature"],
    "Nadzieje": [
        "hope", "wish", "would like", "i'd like", "i want", "please add", 
        "looking forward", "can't wait", "excited for", "plan to", "waiting for"]
}

print(expectation_categories)

# poszerzanie ręcznie stworzonego oczekiwań słownika o często występujące w recenzjach słowa
all_words = Counter()
for tokens in df["tokens"]:
    all_words.update(tokens)

most_common_words = all_words.most_common(400)

def auto_assign_exp_category(word, expectation_categories):
    word_lower = word.lower()
    for category, fragments in expectation_categories.items():
        for frag in fragments:
            frag_lower = frag.lower()
            if len(frag_lower) < 4:
                continue

            if frag_lower in word_lower:
                return category
    return None

new_exp_keywords = {}
for word, count in most_common_words:
    cat = auto_assign_exp_category(word, expectation_categories)
    if cat:
        if word not in expectation_categories[cat]:
            new_exp_keywords.setdefault(cat, []).append(word)

for cat, words in new_exp_keywords.items():
    expectation_categories[cat].extend(words)

print("Nowe słowa dodane do słownika oczekiwań:")
print(new_exp_keywords)

{'DLC': ['dlc', 'expansion', 'expansion pack', 'more content', 'new content', 'addon', 'add-on', 'bonus content', 'season pass'], 'Sequel': ['sequel', 'next game', 'next one', 'bg4', "baldur's gate 4", 'follow-up', 'continuation', 'trilogy', 'future game'], 'Patchowanie': ['fix', 'patch', 'update', 'needs fixing', 'needs update', 'glitch', 'bug fix', 'crash fix', 'optimization', 'optimize', 'repair', 'broken'], 'Co-op': ['multiplayer', 'coop', 'co-op', 'online', 'pvp', 'pve', 'crossplay', 'play with friends'], 'Mechanika': ['new game plus', 'ng+', 'transmog', 'customization', 'photo mode', 'difficulty', 'hard mode', 'easy mode', 'accessibility', 'feature'], 'Nadzieje': ['hope', 'wish', 'would like', "i'd like", 'i want', 'please add', 'looking forward', "can't wait", 'excited for', 'plan to', 'waiting for']}
Nowe słowa dodane do słownika oczekiwań:
{}


In [16]:
# zapis do nowego csv - plik z wszystkimi wynikami analizy sentymentu

df.to_csv("english_bg3_reviews_with_sentiment_full.csv", sep=";", index=False, encoding='utf-8')

In [17]:
df.head()

,review_id,review,voted_up,author_steamid,date_str,review_length,review_word_count,clean_review,is_spam_label,num_emojis,low_quality,is_spam,tokens,sent_vader,sent_vader_opt,bert_sentiment,sent_flair,review_category,future_expectations
0,209490463,The greatest game of all time. It simply is. \r\n,True,76561198842852247,2025-11-17 20:17:24,43,9,the greatest game of all time it simply is,0,0,0,0,"[the, greatest, game, all, time, simply]",1,1,1,1,Unspecified,0
1,209488275,The dude trying to bite me at night was messed...,True,76561198335095296,2025-11-17 19:45:37,55,12,the dude trying to bite me at night was messed...,0,0,0,0,"[the, dude, trying, bite, night, was, messed, ...",0,0,0,0,Unspecified,0
2,209488128,"This is probably a great solo game, but my hop...",False,76561198031957731,2025-11-17 19:43:28,131,29,this is probably a great solo game but my hope...,0,0,0,0,"[this, probably, great, solo, game, but, hope,...",1,1,0,0,"balans, mechanika, soundtrack",1
3,209487979,One of the best RPG's of the last decade,True,76561198140742800,2025-11-17 19:41:16,39,9,one of the best rpgs of the last decade,0,0,0,0,"[one, the, best, rpgs, the, last, decade]",1,1,1,1,Unspecified,0
4,209482559,Definitely one of the best games I've played i...,True,76561198024036703,2025-11-17 18:19:29,61,12,definitely one of the best games ive played in...,0,0,0,0,"[definitely, one, the, best, games, ive, playe...",1,1,1,1,Unspecified,0


In [23]:
# porównujemy wykorzystane wcześniej modele:
# 1) VADER w formie podstawowej
# 2) VADER w wersji zoptymalizowanej
# 3) FLAIR
# 4) BERT

from sklearn.metrics import accuracy_score, f1_score

model_columns = [
    "sent_vader",
    "sent_vader_opt",
    "sent_flair",
    "bert_sentiment"
]

results = {}

for col in model_columns:
    preds = df[col].dropna()
    y_true = df.loc[preds.index, "voted_up"].astype(int)

    acc = accuracy_score(y_true, preds)
    f1 = f1_score(y_true, preds, average="macro")

    results[col] = {
        "accuracy": acc,
        "f1_macro": f1
    }

# wyniki porównania metryk
print("\n=== METRYKI MODELI SENTYMENTU ===")
for model, metrics in results.items():
    print(f"{model}: accuracy={metrics['accuracy']:.4f}, f1_macro={metrics['f1_macro']:.4f}")

# best model 
best_model = max(results, key=lambda m: results[m]["f1_macro"])

print("Najlepszy model analizowania sentymentu dla recenzji pochodzących z portalu Steam to:")
print(best_model, results[best_model])

# usuwamy kolumny z wynikami innych modeli niż wybrany best model
cols_to_drop = [col for col in model_columns if col != best_model]
df = df.drop(columns=cols_to_drop)

KeyError: 'sent_vader'

In [7]:
# identyfikacja oczekiwań użytkowników
# funkcja sprawdza czy w treści recenzji są określone w słowniku oczekiwania
def contains_expectation(text):
    text = str(text).lower()
    for keywords in expectation_categories.values():
        for kw in keywords:
            if kw in text:
                return True
    return False

# funkcja przypisuje konkretną kategorię do ramki danych
def assign_expectation_category(text):
    text = str(text).lower()
    found_categories = set()
    
    for category, keywords in expectation_categories.items():
        for kw in keywords:
            if kw in text:
                found_categories.add(category)
    return ", ".join(sorted(found_categories)) if found_categories else "Unspecified"

# dodawanie kolumny do df
df["has_expectation"] = df["clean_review"].apply(contains_expectation)

# podzbiór z oczekiwaniami i zliczanie według kategorii
expectation_df = df[df["has_expectation"] == True].copy()
expectation_df["expectation_category"] = expectation_df["clean_review"].apply(assign_expectation_category)
expectation_summary = expectation_df["expectation_category"].str.split(", ").explode().value_counts()

print("Oczekiwania użytkowników według kategorii to m.in.:")
print(expectation_summary)


Oczekiwania użytkowników według kategorii to m.in.:
expectation_category
Nadzieje       3778
Patchowanie    2610
Co-op          1803
Mechanika      1737
DLC             891
Sequel          401
Name: count, dtype: int64


In [20]:
# zapis do nowego csv - plik z wynikami tylko best model

df.to_csv("english_bg3_reviews_with_sentiment_best_model.csv", sep=";", index=False, encoding='utf-8')

In [27]:
# ścieżka zapisu modelu BERT
save_directory = "./models/bert_sentiment"

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# zapisuje konfiguracje (informacje o hiperparametrach modelu), wagi (modelu)
# oraz tokenizer (tłumacz modelu)
tokenizer.save_pretrained(save_directory)
model.save_pretrained(save_directory)

print("Model BERT zapisany w folderze")

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 990d3b8d-2e84-4e1f-8622-bfe044114e8b)')' thrown while requesting HEAD https://huggingface.co/nlptown/bert-base-multilingual-uncased-sentiment/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


Model BERT zapisany w folderze


In [26]:
# zapis Flair
models_dir = './models'
output_path = os.path.join(models_dir, 'flair_sentiment.pt')
flair_model.save(output_path)

print(f"Flair zapisano jako '{output_path}'.")

Flair zapisano jako './models\flair_sentiment.pt'.


In [28]:
df.count()

review_id              53048
review                 53048
voted_up               53048
author_steamid         53048
date_str               53048
review_length          53048
review_word_count      53048
clean_review           53048
is_spam_label          53048
num_emojis             53048
low_quality            53048
is_spam                53048
tokens                 53048
sent_flair             53048
review_category        53048
future_expectations    53048
has_expectation        53048
dtype: int64